In [36]:
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm 


In [38]:
url_emb_df = pd.read_parquet('data/urls.txt_sberbank-ai_ruElectra-small.parquet')

In [39]:
url_emb_df.head()

,url_host_preprocessed,url_host_embedding
0,,"[0.01859397, 0.28440517, 0.064282626, -0.02738..."
1,uznayonline,"[0.018607048, 0.3694605, 0.059723165, -0.02284..."
2,mebetal,"[0.018624451, 0.21577111, 0.05554186, -0.02540..."
3,belohota,"[0.01864303, 0.22606, 0.049342472, -0.02587692..."
4,toratarot.forum2x2,"[0.0186494, 0.01898303, 0.047724646, -0.023608..."


In [40]:
url_id_df = pd.read_csv('data/url_host_preprocesed_mapping.csv')

In [50]:
url_id_df[url_id_df['_orig_url_host_preprocesed'].isna()]

,_orig_url_host_preprocesed,url_host_preprocesed
104,NaN,106


In [52]:
url_id_df['_orig_url_host_preprocesed'] = url_id_df['_orig_url_host_preprocesed'].fillna('')

In [115]:
url_id_df.head()

,_orig_url_host_preprocesed,url_host_preprocesed
0,googleads.g.doubleclick,2
1,yandex,3
2,i.ytimg,4
3,vk,5
4,avatars.mds.yandex,6


In [116]:
url_id_df_renamed = url_id_df.rename(
    columns={'url_host_preprocesed': 'ptls_id', 
             '_orig_url_host_preprocesed': 'url_host_preprocessed'}
)

In [117]:
url_id_df_renamed.head()

,url_host_preprocessed,ptls_id
0,googleads.g.doubleclick,2
1,yandex,3
2,i.ytimg,4
3,vk,5
4,avatars.mds.yandex,6


In [119]:
url_id_df_renamed[url_id_df_renamed['url_host_preprocessed'].isna()]

,url_host_preprocessed,ptls_id


In [123]:
url_emb_df.head()

,url_host_preprocessed,url_host_embedding
0,,"[0.01859397, 0.28440517, 0.064282626, -0.02738..."
1,uznayonline,"[0.018607048, 0.3694605, 0.059723165, -0.02284..."
2,mebetal,"[0.018624451, 0.21577111, 0.05554186, -0.02540..."
3,belohota,"[0.01864303, 0.22606, 0.049342472, -0.02587692..."
4,toratarot.forum2x2,"[0.0186494, 0.01898303, 0.047724646, -0.023608..."


In [124]:
emb_id_df = pd.merge(left = url_emb_df, right = url_id_df_renamed,  on='url_host_preprocessed')

In [125]:
emb_id_df.head()

,url_host_preprocessed,url_host_embedding,ptls_id
0,,"[0.01859397, 0.28440517, 0.064282626, -0.02738...",106
1,uznayonline,"[0.018607048, 0.3694605, 0.059723165, -0.02284...",56281
2,mebetal,"[0.018624451, 0.21577111, 0.05554186, -0.02540...",97434
3,belohota,"[0.01864303, 0.22606, 0.049342472, -0.02587692...",86883
4,toratarot.forum2x2,"[0.0186494, 0.01898303, 0.047724646, -0.023608...",166080


In [130]:
emb_id_df.isna().values.any()

False

In [131]:
emb_id_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196693 entries, 0 to 196692
Data columns (total 3 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   url_host_preprocessed  196693 non-null  object
 1   url_host_embedding     196693 non-null  object
 2   ptls_id                196693 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 4.5+ MB


In [132]:
emb_id_df[emb_id_df['url_host_preprocessed'] == '']

,url_host_preprocessed,url_host_embedding,ptls_id
0,,"[0.01859397, 0.28440517, 0.064282626, -0.02738...",106


In [133]:

all_embeddings =  np.vstack(emb_id_df['url_host_embedding'].values)
n_embeddings, embedding_size = all_embeddings.shape
all_ids = emb_id_df['ptls_id'].values
max_embedding_id = emb_id_df['ptls_id'].max()
min_embedding_id = emb_id_df['ptls_id'].min()

embedding_size, max_embedding_id, min_embedding_id == 2, n_embeddings == max_embedding_id - min_embedding_id + 1

(256, 196694, True, True)

In [71]:
max_embedding_id

196694

In [69]:
n_embeddings

196693

In [72]:
for i in tqdm(range(n_embeddings)):
    assert np.all(emb_id_df[emb_id_df['ptls_id'] == all_ids[i]]['url_host_embedding'].values[0]    ==    all_embeddings[i])

100%|██████████| 196693/196693 [01:16<00:00, 2561.71it/s]


In [137]:
url_host_embeddings_ptls_ids = torch.zeros((max_embedding_id+1, embedding_size), requires_grad=False)
url_host_embeddings_ptls_ids[all_ids] = torch.tensor(all_embeddings)

In [138]:
torch.save(url_host_embeddings_ptls_ids, 'data/url_host_embeddings_ptls_ids.pt')

In [139]:
zeros_row = torch.zeros(embedding_size)
entierly_zero_rows = (url_host_embeddings_ptls_ids == zeros_row).all(dim=1).nonzero()

  0%|          | 0/196695 [00:00<?, ?it/s]

100%|██████████| 196695/196695 [00:01<00:00, 138256.03it/s]


In [140]:
entierly_zero_rows

[0, 1]

# Create url embeddings where emb[i] is embedding of i-th GRAPH item

those are graph_item)ids with offset equal to min graph_item_id

In [141]:
import torch

In [142]:
url_host_embeddings_ptls_ids = torch.load('data/url_host_embeddings_ptls_ids.pt')

In [143]:
item_id2train_graph_id = torch.load('data/graphs/weighted_train_graph_has_test_items/item_id2train_graph_id.pt')

In [144]:
item_id2train_graph_id

tensor([     0,      0, 388317,  ..., 585009, 580544, 580545])

In [145]:
torch.sum(item_id2train_graph_id == 0)

tensor(2)

In [146]:
# Since node_id 0 is a user, all indexes i such that item_id2train_graph_id[i] == 0 are invalid indexes.

train_graph_id_to_item_id = {int(v): k for k, v in enumerate(item_id2train_graph_id) if int(v) != 0}

In [147]:
min_graph_item_id = min(train_graph_id_to_item_id.keys())
min_graph_item_id

388317

In [148]:
url_host_embeddings_ptls_ids.shape

torch.Size([196695, 256])

In [149]:
item_id2train_graph_id.shape

torch.Size([196695])

In [150]:
max(train_graph_id_to_item_id.keys()) + 1

585010

In [151]:
from tqdm import tqdm
n_rows = max(train_graph_id_to_item_id.keys()) + 1 - min_graph_item_id
emb_size = url_host_embeddings_ptls_ids.shape[1]

url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset = torch.zeros(n_rows, emb_size)
for k, v in tqdm(train_graph_id_to_item_id.items()):
    url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset[k - min_graph_item_id] = url_host_embeddings_ptls_ids[v]

  0%|          | 0/196693 [00:00<?, ?it/s]

100%|██████████| 196693/196693 [00:01<00:00, 173888.88it/s]


In [152]:
url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset.shape

torch.Size([196693, 256])

In [153]:
entierly_zero_rows = []
for i in tqdm(range(url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset.shape[0])):
    if torch.equal(url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset[i], torch.zeros(emb_size)):
        entierly_zero_rows.append(i)

100%|██████████| 196693/196693 [00:01<00:00, 137612.09it/s]


In [154]:
len(entierly_zero_rows)

0

In [155]:
entierly_zero_rows

[]

In [156]:
train_graph_id_to_item_id[104+min_graph_item_id]

106

In [157]:
torch.save(url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset, 
           'data/url_host_embeddings_graph_ids__train_graph_with_added_items_ids__has_offset.pt')